### Load Pdf File

In [2]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader,
    UnstructuredPDFLoader
)

In [3]:
### pypdfloader
print("PyPdfloader")

try:
    pypdf_loader = PyPDFLoader("data/pdf/Impact Two, 1-80.pdf")
    pypdf_docs = pypdf_loader.load()
    print(pypdf_docs)
    print(f" Loaded {len(pypdf_docs)} pages")
    print(f" Page 1 content {pypdf_docs[0].page_content[:100]}...")
    print(f" Metadata: {pypdf_docs[0].metadata}")

except Exception as e:
    print(f"Error: {e}")

PyPdfloader
[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-05-12T04:52:31+00:00', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages': 72, 'page': 0, 'page_label': '1'}, page_content=''), Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-05-12T04:52:31+00:00', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages': 72, 'page': 1, 'page_label': '2'}, page_content=''), Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-05-12T04:52:31+00:00', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages': 72, 'page': 2, 'page_label': '3'}, page_content=''), Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-05-12T04:52:31+00:00', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages': 72, 'page': 3, 'page_label': '4'}, page_content=''), Document(metadata={'producer': 'iLovePDF', 'creator

In [5]:
### PyMupdfloader (fast and accurate)
print("PyMuPdfloader")

try:
    pymupdf_loader = PyMuPDFLoader("data/pdf/Impact Two, 1-80.pdf")
    pymupdf_docs = pymupdf_loader.load()
    print(pymupdf_docs)
    print(f" Loaded {len(pymupdf_docs)} pages")
    print(" Including detailed metadata ")
    print(pymupdf_docs)

except Exception as e:
    print(f"Error: {e}")

PyMuPdfloader
[Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'file_path': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages': 72, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-05-12T04:52:31+00:00', 'trapped': '', 'modDate': 'D:20220512045231Z', 'creationDate': '', 'page': 0}, page_content=''), Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'file_path': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages': 72, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2022-05-12T04:52:31+00:00', 'trapped': '', 'modDate': 'D:20220512045231Z', 'creationDate': '', 'page': 1}, page_content=''), Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': 'data/pdf/Impact Two, 1-80.pdf', 'file_path': 'data/pdf/Impact Two, 1-80.pdf', 'total_pages':

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [39]:
from langchain_core.documents import Document
from typing import List
class SmartPDFProcessor:
    """Advanced pdf processing with error handling"""
    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size=chunk_size
        self.chunk_overlap=chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            # separators=["\n\n", "\n", " ", ""]
            separators=[" "]

        )

    def process_pdf(self, pdf_path:str)->List[Document]:
        """process pdf with smart chunking and meta data enhancment"""

        #load pdf 

        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        ##process each page 

        processed_chunk = []

        for page_num, page in enumerate(pages):
            ##clean text
            cleaned_text = self._clean_text(page.page_content)

            # skip nearly empty pages
            if len(cleaned_text.strip()) < 10:
                continue

            # create chunk with enhaced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page":page_num+1,
                    "total_pages":len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text)

                }]
            )
            processed_chunk.extend(chunks)

        return processed_chunk
    def _clean_text(self, text:str) -> str:
        """clean extraced text"""
        #remove excessive whitspace
        text = " ".join(text.split())

        #fixed common pdf extraction issues
        text = text.replace("f1", "fi")
        text = text.replace("f2", "fl")

        return text




In [40]:
preprocessor = SmartPDFProcessor()

In [41]:
preprocessor

In [42]:
##process a pdf if available
try:
    smart_chunks=preprocessor.process_pdf("data/pdf/ai.pdf")
    print(f"processed into {len(smart_chunks)} smart chunks")

    #show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata")
        for key, value in smart_chunks[0].metadata.items():
            print(f" {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

            

processed into 129 smart chunks

Sample chunk metadata
 producer: Adobe PDF Library 17.0
 creator: Adobe InDesign 19.0 (Macintosh)
 creationdate: 2024-03-06T16:39:02+00:00
 moddate: 2024-03-06T16:39:08+00:00
 trapped: /False
 source: data/pdf/ai.pdf
 total_pages: 44
 page: 1
 page_label: 1
 chunk_method: smart_pdf_processor
 char_count: 168
